# Example: Contract Review Agent

A `dspy.ReAct` agent that reviews real commercial contracts using the CUAD dataset.
Built to accompany the *What Your Agent Is Doing When You're Not Watching* blog post on [matlog.dev](https://matlog.dev/posts/ai-agent-observability).

**What makes this interesting:** the agent must decide, at each step, which tool to reach for —
and the tasks are designed to force it to iterate across all five tools before it can finish.

**Seven tools:**
1. `list_contracts(filter_term)` — discover available contracts in the contract store
2. `search_clauses(contract_id, clause_type)` — retrieve a specific clause from a contract
3. `assess_clause(clause_text, playbook_rule)` — evaluate a clause against a legal standard
4. `web_search(query)` — Brave Search API for market standards and regulatory guidance
5. `fetch_url(url)` — read the full text of a specific webpage
6. `find_contracts_with_clause(clause_type)` — find all contracts in the store that contain a given clause type
7. `list_clause_types()` — discover which clause types are available in the contract store

**Three tasks (increasing complexity):**
- Task 1: Governing law review — 3 tool calls expected
- Task 2: Liability cap vs. market standard — 4–5 tool calls expected
- Task 3: GDPR compliance gap analysis — 6+ tool calls expected

**Requires:** `BRAVE_API_KEY` in your `.env` file (get one at https://brave.com/search/api/)

---
## Setup — LM + MLflow

In [5]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
warnings.filterwarnings("ignore", message="You are sending unauthenticated requests to the HF Hub")
import difflib
import dspy
import logging
import mlflow
import os
import re
from collections import Counter, defaultdict
from dotenv import load_dotenv
from datasets import load_dataset
from typing import Literal
import requests
from IPython.display import Markdown, display



load_dotenv()

lm = dspy.LM(
    provider=os.getenv("PROVIDER_ENDPOINT"),
    api_key=os.getenv("PROVIDER_API_KEY"),
    model=os.getenv("MODEL")
)

dspy.configure(lm=lm)

# ── MLflow — tracing is opt-in via env var, no-op if not set ─────────────────
tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5001')
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment('contract-review-agent')
mlflow.dspy.autolog()  # Captures every DSPy call, tool invocation, and token count

# Suppress verbose INFO output from MLflow and its tracing dependencies.
# Traces are still recorded in full — this only affects console noise.
for logger_name in ('mlflow', 'opentelemetry', 'opentelemetry.sdk', 'dspy'):
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# ── Brave Search ──────────────────────────────────────────────────────────────
BRAVE_API_KEY = os.getenv('BRAVE_API_KEY')
if not BRAVE_API_KEY:
    print('⚠  BRAVE_API_KEY not set — web_search() will return a placeholder.')
    print('   Get a free key at https://brave.com/search/api/ and add it to .env')

print(f'LM:     {lm.model}')
print(f'MLflow: {tracking_uri}  (experiment: contract-review-agent)')
print(f'Brave:  {"configured" if BRAVE_API_KEY else "not configured"}')

LM:     openrouter/moonshotai/kimi-k2.5
MLflow: http://localhost:5001  (experiment: contract-review-agent)
Brave:  configured


---
## 1. Loading CUAD

The **Contract Understanding Atticus Dataset (CUAD)** contains 509 commercial contracts
labeled for 41 clause types — NDA, software licensing, service agreements, M&A exhibits.
All contracts are public-domain filings. No client data.

The dataset is SQuAD-style: each row is one (contract, clause_type_question) pair.
We pivot it into a lookup dict: `contract_id → {clause_type → clause_text}`.

First run downloads ~500 MB to the HuggingFace cache. Subsequent runs are instant.

In [6]:
print('Loading dataset... (first run downloads to HuggingFace cache)')
dataset = load_dataset(
    'dvgodoy/CUAD_v1_Contract_Understanding_clause_classification',
    split='train'
)
print(f'Loaded {len(dataset)} clause entries')
print(f'\nColumns: {dataset.column_names}')
print(f'\nUnique clause types: {len(set(dataset["label"]))}')
print('\nSample labels:')
for label in list(set(dataset['label']))[:10]:
    print(f'  {label}')

Loading dataset... (first run downloads to HuggingFace cache)
Loaded 13155 clause entries

Columns: ['file_name', 'clause', 'pages', 'class_id', 'label', 'start_at', 'end_at']

Unique clause types: 41

Sample labels:
  Change Of Control
  Notice Period To Terminate Renewal
  Cap On Liability
  Insurance
  Competitive Restriction Exception
  No-Solicit Of Employees
  Unlimited/All-You-Can-Eat-License
  Volume Restriction
  Source Code Escrow
  Liquidated Damages


#### Normalize Data
Normalize clause labels to match our tool interface.
The dataset uses human-readable labels like "Most Favored Nation".
We map them to snake_case keys for the tool interface.

In [3]:
LABEL_TO_KEY = {
    # ── Exact dataset matches ──────────────────────────────────────────────────
    'Governing Law':                        'governing_law',
    'Non-Compete':                          'non_compete',
    'Termination For Convenience':          'termination_for_convenience',
    'Audit Rights':                         'audit_rights',
    'Most Favored Nation':                  'most_favored_nation',
    'Change Of Control':                    'change_of_control',
    'Exclusivity':                          'exclusivity',
    'Price Restrictions':                   'price_restrictions',
    'Minimum Commitment':                   'minimum_commitment',
    'Volume Restriction':                   'volume_restriction',
    'Insurance':                            'insurance',
    'Covenant Not To Sue':                  'covenant_not_to_sue',
    'Third Party Beneficiary':              'third_party_beneficiary',
    'Rofr/Rofo/Rofn':                      'rofr_rofo_rofn',
    'Uncapped Liability':                   'uncapped_liability',
    'License Grant':                        'license_grant',
    'Source Code Escrow':                   'source_code_escrow',
    'Post-Termination Services':            'post_termination_services',
    'Cap On Liability':                     'cap_on_liability',
    # ── Renamed from original CUAD questions ──────────────────────────────────
    'Ip Ownership Assignment':              'ip_ownership',
    'Warranty Duration':                    'warranty',
    'Anti-Assignment':                      'assignment',
    'Renewal Term':                         'auto_renewal',
    'Revenue/Profit Sharing':               'revenue_sharing',
    'Joint Ip Ownership':                   'joint_ip_ownership',
    # ── Additional dataset labels ──────────────────────────────────────────────
    'Non-Disparagement':                    'non_disparagement',
    'Liquidated Damages':                   'liquidated_damages',
    'Affiliate License-Licensee':           'affiliate_license_licensee',
    'Affiliate License-Licensor':           'affiliate_license_licensor',
    'Irrevocable Or Perpetual License':     'irrevocable_perpetual_license',
    'Non-Transferable License':             'non_transferable_license',
    'No-Solicit Of Customers':              'no_solicit_customers',
    'No-Solicit Of Employees':              'no_solicit_employees',
    'Competitive Restriction Exception':    'competitive_restriction_exception',
    'Unlimited/All-You-Can-Eat-License':    'unlimited_license',
}

# Case-insensitive lookup used during dataset ingestion.
# Keeps LABEL_TO_KEY human-readable while tolerating label casing variations.
_LABEL_LOOKUP = {k.lower(): v for k, v in LABEL_TO_KEY.items()}


def make_contract_id(file_name: str) -> str:
    """Create a clean, URL-safe contract identifier from a filename.
    Lowercase, non-alphanumeric chars become underscores, max 50 chars.
    """
    base = file_name.replace('.pdf', '').lower()
    return re.sub(r'[^a-z0-9]+', '_', base)[:50].strip('_')

#### Building the Contract Store
In production this would be a database query or vector store lookup.
Here it's a dict: {contract_id: {clause_type: {text, contract_title}}}



In [4]:
contract_store = defaultdict(dict)

for row in dataset:
    clause_type = _LABEL_LOOKUP.get(row["label"].lower())
    if not clause_type:
        continue  # Skip labels we don't map

    contract_id = make_contract_id(row["file_name"])
    contract_title = row["file_name"].replace(".pdf", "")

    # Keep first occurrence only for each clause type per contract
    if clause_type not in contract_store[contract_id]:
        contract_store[contract_id][clause_type] = {
            "text": row["clause"],
            "contract_title": contract_title,
        }

total_clauses = sum(len(v) for v in contract_store.values())

# ── Diagnostic: clause type coverage ─────────────────────────────────────────
clause_counts = Counter(
    clause_type
    for clauses in contract_store.values()
    for clause_type in clauses
)

rows = "\n".join(
    f"| `{k}` | {v} |"
    for k, v in sorted(clause_counts.items(), key=lambda x: -x[1])
)

display(Markdown(
    f"**Contract store ready** — {len(contract_store)} contracts, {total_clauses} clause entries\n\n"
    f"| Clause type | Contracts |\n|---|---|\n{rows}"
))

**Contract store ready** — 456 contracts, 4252 clause entries

| Clause type | Contracts |
|---|---|
| `governing_law` | 425 |
| `assignment` | 371 |
| `cap_on_liability` | 273 |
| `license_grant` | 252 |
| `audit_rights` | 213 |
| `post_termination_services` | 182 |
| `termination_for_convenience` | 180 |
| `exclusivity` | 178 |
| `auto_renewal` | 175 |
| `insurance` | 166 |
| `minimum_commitment` | 163 |
| `revenue_sharing` | 162 |
| `non_transferable_license` | 136 |
| `ip_ownership` | 122 |
| `change_of_control` | 120 |
| `non_compete` | 117 |
| `uncapped_liability` | 111 |
| `covenant_not_to_sue` | 100 |
| `rofr_rofo_rofn` | 85 |
| `volume_restriction` | 81 |
| `competitive_restriction_exception` | 76 |
| `warranty` | 73 |
| `irrevocable_perpetual_license` | 70 |
| `liquidated_damages` | 61 |
| `affiliate_license_licensee` | 59 |
| `no_solicit_employees` | 57 |
| `joint_ip_ownership` | 46 |
| `non_disparagement` | 38 |
| `no_solicit_customers` | 33 |
| `third_party_beneficiary` | 32 |
| `most_favored_nation` | 28 |
| `affiliate_license_licensor` | 23 |
| `unlimited_license` | 17 |
| `price_restrictions` | 15 |
| `source_code_escrow` | 12 |

---
## 2. Tool Definitions

DSPy `ReAct` tools are plain Python functions. The agent reads the function name, type annotations,
and docstring to decide when and how to call each tool — which means **the docstring is
load-bearing**. A vague docstring produces poor tool selection decisions.

Good tool docstrings answer:
- What does this tool do? (one sentence)
- When should the agent call it?
- What are the valid inputs?
- What does it return?

#### Tool 1: list_contracts
Discovery tool — the agent calls this first when it doesn't know which
contract to review. Supports keyword filtering so the agent can narrow down
by contract type (e.g. 'subscription', 'license', 'service').


In [5]:
def list_contracts(filter_term: str = '') -> str:
    """List available contracts in the contract store.

    Use this tool first when you need to find a contract to review.
    Returns contract IDs that can be passed to search_clauses().

    Args:
        filter_term: Optional keyword to filter by (e.g. 'subscription', 'license',
                     'service', 'software', 'nda'). Pass an empty string for all contracts.

    Returns a list of contract IDs. Always use an ID from this list with search_clauses().
    """
    if filter_term:
        matches = [cid for cid in contract_store if filter_term.lower() in cid]
        label = f"matching '{filter_term}'"
    else:
        matches = list(contract_store.keys())
        label = 'total'

    display = matches[:20]
    lines = [f'  {cid}  (clause types: {len(contract_store[cid])})' for cid in display]
    return (
        f'Found {len(matches)} contracts {label} (showing first {len(display)}):\n'
        + '\n'.join(lines)
    )

# Quick smoke test
print(list_contracts('software')[:500])

Found 4 contracts matching 'software' (showing first 4):
  inktomicorp_06_08_1998_ex_10_14_software_hosting_a  (clause types: 15)
  sfgfinancialcorp_05_12_2009_ex_10_1_software_licen  (clause types: 10)
  hyperionsoftwarecorp_09_28_1994_ex_10_47_exclusive  (clause types: 14)
  summafourinc_06_19_1998_ex_10_3_software_license_a  (clause types: 13)


#### Tool 2: search_clauses
The main retrieval tool. Fuzzy-matches on both `contract_id` and `clause_type`
using `difflib.get_close_matches` — so the agent doesn't need exact spelling.

If the clause isn't in the chosen contract, the error message points to
`find_contracts_with_clause()` (Tool 6) rather than leaving the agent to guess.

In [6]:
def search_clauses(contract_id: str, clause_type: str) -> str:
    """Retrieve a specific clause from a contract.

    Use find_contracts_with_clause() first if you don't know which contracts
    contain the clause type you're looking for.

    Args:
        contract_id: A contract ID from list_contracts() or find_contracts_with_clause().
                     Partial match supported — 'atlassian' finds 'atlassian_subscriber_agreement__'.
        clause_type: One of: governing_law, indemnification, limitation_of_liability,
                     non_compete, termination_for_convenience, ip_ownership, confidentiality,
                     warranty, assignment, audit_rights, most_favored_nation,
                     change_of_control, arbitration, auto_renewal, exclusivity,
                     price_restrictions, revenue_sharing, minimum_commitment, insurance,
                     license_grant, cap_on_liability.
                     Fuzzy matching is supported — minor spelling variations are resolved
                     automatically.

    Returns the clause text, or a helpful message if not found.
    """
    cid_lower = contract_id.lower().replace(' ', '_')

    # Exact match first, then partial
    if cid_lower in contract_store:
        matched_id = cid_lower
    else:
        candidates = [k for k in contract_store if cid_lower in k or k.startswith(cid_lower[:10])]
        if not candidates:
            return (
                f"Contract '{contract_id}' not found. "
                f"Try list_contracts('{contract_id.split('_')[0]}') to find similar names."
            )
        matched_id = candidates[0]

    clauses = contract_store[matched_id]

    # Resolve clause_type — exact match, then fuzzy
    if clause_type in clauses:
        matched_type = clause_type
    else:
        close = difflib.get_close_matches(clause_type, clauses.keys(), n=1, cutoff=0.6)
        if close:
            matched_type = close[0]
        else:
            available = sorted(clauses.keys())
            return (
                f"Clause type '{clause_type}' not found in '{matched_id}'.\n"
                f"Available clause types for this contract: {available}\n"
                f"Tip: use find_contracts_with_clause('{clause_type}') to find a contract that has it."
            )

    note = f" (fuzzy-matched from '{clause_type}')" if matched_type != clause_type else ""
    entry = clauses[matched_type]
    return (
        f"[{matched_type.upper()}{note} from '{entry['contract_title'][:50]}'\n"
        f"Contract ID used: {matched_id}]\n\n"
        f"{entry['text']}"
    )

# Smoke test
Markdown(search_clauses('license', 'governing_law'))

[GOVERNING_LAW from 'CORIOINC_07_20_2000-EX-10.5-LICENSE AND HOSTING AG'
Contract ID used: corioinc_07_20_2000_ex_10_5_license_and_hosting_ag]

This Agreement shall be governed by the laws of the State of California, USA, excluding conflict of laws provisions and excluding the 1980 United Nations Convention on Contracts for the International Sale of Goods.

#### Tool 3: assess_clause
This tool is unusual: it makes an LM call internally (a DSPy ChainOfThought).
The outer agent (ReAct) calls it as a tool, which triggers a nested LM call.
DSPy handles this cleanly — the nested call appears as a child span in the MLflow trace.

Why this pattern instead of inline reasoning?
The outer ReAct agent's job is to plan and orchestrate (decide what to review and against
which standard). The inner ChainOfThought's job is deep clause analysis. Separating them
keeps each LM call focused on one task and makes the trace easier to read.


In [7]:
class AssessSignature(dspy.Signature):
    clause_text: str = dspy.InputField()
    playbook_rule: str = dspy.InputField()
    verdict: Literal['compliant', 'non_compliant', 'requires_review'] = dspy.OutputField()

_assessor = dspy.ChainOfThought(AssessSignature)

def assess_clause(clause_text: str, playbook_rule: str) -> str:
    """Assess whether a contract clause complies with a given playbook rule.

    Args:
        clause_text: The verbatim clause text.
        playbook_rule: A plain-English statement of what the standard position requires.
                       Be specific — e.g. 'Liability cap must not exceed 12 months of fees.
                       Must exclude consequential damages. No carve-outs for IP infringement.'

    Returns a verdict (compliant / non_compliant / requires_review), rationale, and a
    suggested revision if the clause is non-compliant.
    """
    if not clause_text or clause_text.strip() in ('', 'not found', 'None'):
        return "Cannot assess: no clause text provided. Call search_clauses() first."

    result = _assessor(clause_text=clause_text, playbook_rule=playbook_rule)
    return (
        f"Verdict: {result.verdict.upper()}\n\n"
        f"Rationale: {result.rationale}\n\n"
        f"Suggested revision: {result.suggested_revision}"
    )

print('assess_clause defined (uses ChainOfThought internally — makes a nested LM call)')

assess_clause defined (uses ChainOfThought internally — makes a nested LM call)


#### Tool 4: web_search
The research tool. Brave Search API — no tracking, no personalization.
The agent should use this when it needs market standards, regulatory guidance,
or case law context it can't derive from the contract text alone.

In production, you should not rely on web search for critical information retrieval — the results can be inconsistent, and you don't want your agent's performance to hinge on the quality of search results. Instead, use web search as a last resort for context you can't get from your internal knowledge base.


In [8]:
def web_search(query: str, num_results: int = 5) -> str:
    """Search the web using Brave Search for legal research and market standards.

    Use this when you need:
    - Current market standards (e.g. 'SaaS liability cap standard 2025')
    - Regulatory guidance (e.g. 'GDPR data processing agreement requirements ICO')
    - Case law context (e.g. 'Delaware governing law clause enforcement')
    - Industry practice surveys (e.g. 'IACCM indemnification market data 2024')

    Args:
        query: A specific, keyword-rich search query. Shorter queries work better.
        num_results: Number of results to return (1–10, default 5).

    Returns search results with titles, URLs, and snippets. Use fetch_url() to
    read the full content of any result that looks promising.
    """
    if not BRAVE_API_KEY:
        return (
            f"[BRAVE_API_KEY not configured — simulated result for query: '{query}']\n\n"
            f"1. Market Standard for {query}\n"
            f"   URL: https://example.com/legal-standards\n"
            f"   Industry surveys suggest standard practice for this topic requires "
            f"specific provisions addressing the key legal concerns raised by the query."
        )

    try:
        response = requests.get(
            'https://api.search.brave.com/res/v1/web/search',
            headers={
                'Accept': 'application/json',
                'Accept-Encoding': 'gzip',
                'X-Subscription-Token': BRAVE_API_KEY,
            },
            params={'q': query, 'count': min(int(num_results), 10)},
            timeout=10,
        )
        response.raise_for_status()
        results = response.json().get('web', {}).get('results', [])

        if not results:
            return f"No results found for: '{query}'. Try a broader or different query."

        formatted = []
        for i, r in enumerate(results[:num_results], 1):
            formatted.append(
                f"{i}. {r.get('title', 'No title')}\n"
                f"   URL: {r.get('url', '')}\n"
                f"   {r.get('description', 'No description available')}"
            )
        return '\n\n'.join(formatted)

    except requests.exceptions.Timeout:
        return f"Search timed out for '{query}'. Try a simpler query."
    except requests.exceptions.HTTPError as e:
        return f"Search API error {e.response.status_code}. Check your BRAVE_API_KEY."
    except Exception as e:
        return f"Search error: {e}"

# Smoke test
Markdown(web_search('governing law clause commercial contract Delaware', num_results=2))

1. Governing Law Jurisdiction Contract Clause Examples | Business Contracts | Justia
   URL: https://contracts.justia.com/contract-clauses/governing-law-jurisdiction/
   This page contains Governing Law Jurisdiction clauses in business contracts and legal agreements. We have organized these clauses into groups of similarly worded clauses. Governing Law Jurisdiction. This Agreement, the rights and obligations of the parties hereto, and any claims or disputes relating thereto, shall be governed by and construed in accordance with the laws of the State of Delaware without regard to its choice of law provisions.

2. Delaware Code, Title 6, Chapter 27
   URL: https://delcode.delaware.gov/title6/c027/sc01/index.html
   Laws, c. 175, § 1; ... (a) The parties to any contract, agreement or other undertaking, contingent or otherwise, may agree in writing that <strong>the contract, agreement or other undertaking shall be governed by or construed under the laws of this State, without regard to principles of conflict of laws</strong>, ...

#### Tool 5: fetch_url
Read the full content of a specific page. The agent uses this after web_search()
returns an interesting URL it wants to read in full.

Very naive HTML stripping uses only stdlib (re) — no BeautifulSoup dependency needed.
Truncates at max_chars to avoid blowing the context window.


In [9]:

def fetch_url(url: str, max_chars: int = 3000) -> str:
    """Fetch the text content of a webpage.

    Use this to read web pages in full — 
    e.g. an ICO guidance page, a legal blog post, or a court decision.

    Args:
        url: Full URL starting with http:// or https://.
        max_chars: Maximum characters to return (default 3000 — fits in context window).
                   Increase to 6000 for detailed regulatory documents.

    Returns cleaned text content of the page. Script and style elements are stripped.
    """
    if not url.startswith(('http://', 'https://')):
        return f"Invalid URL: must start with http:// or https://. Got: {url}"

    try:
        response = requests.get(
            url,
            headers={'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'},
            timeout=15,
        )
        response.raise_for_status()

        text = response.text
        # Strip scripts and styles first (they contain code, not content)
        text = re.sub(r'<script[^>]*>.*?</script>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
        text = re.sub(r'<style[^>]*>.*?</style>', ' ', text, flags=re.DOTALL | re.IGNORECASE)
        # Strip remaining HTML tags
        text = re.sub(r'<[^>]+>', ' ', text)
        # Decode common HTML entities
        for entity, char in [('&amp;', '&'), ('&lt;', '<'), ('&gt;', '>'), ('&nbsp;', ' '), ('&quot;', '"')]:
            text = text.replace(entity, char)
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        if len(text) > max_chars:
            return (
                text[:max_chars]
                + f'\n\n[Content truncated at {max_chars:,} chars. '
                + f'Full page was {len(text):,} chars. '
                + 'Call again with max_chars=6000 for more.]'
            )
        return text

    except requests.exceptions.Timeout:
        return f"Timeout fetching {url}. The server is slow — try a different URL."
    except requests.exceptions.HTTPError as e:
        return f"HTTP {e.response.status_code} fetching {url}."
    except Exception as e:
        return f"Error fetching {url}: {type(e).__name__}: {e}"

print('fetch_url defined (HTML stripping via stdlib re — no external dependencies)')
Markdown(fetch_url('https://www.matlog.dev')[:300])

fetch_url defined (HTML stripping via stdlib re — no external dependencies)


matlog.dev - Tales of a Lawyer Turned Developer { M } atlog Home Archive About From Law to Code Tales of a Lawyer Turned Developer Hi, I'm Matthias. I practiced law for 8 years while teaching myself to code. Now I build automation tools for legal and tax workflows combining legal and technical exper

#### Tool 6: find_contracts_with_clause
Inverse lookup — instead of "give me clause X from contract Y", this answers
"which contracts actually have clause X?"

In [10]:
def find_contracts_with_clause(clause_type: str) -> str:
    """Find all contracts in the store that contain a given clause type.

    Use this when you need to locate a contract that has a specific clause —
    especially before calling search_clauses() on a clause-first task.
    Fuzzy matching on clause_type handles minor spelling variations.

    Args:
        clause_type: The clause type to search for. Examples: governing_law,
                     limitation_of_liability, cap_on_liability, confidentiality,
                     indemnification, ip_ownership, termination_for_convenience.
                     Fuzzy matching is supported.

    Returns a list of contract IDs that contain this clause type.
    Pass any returned ID directly to search_clauses() to retrieve the text.
    """
    all_keys = {key for clauses in contract_store.values() for key in clauses}

    # Exact match first, then fuzzy
    if clause_type in all_keys:
        matched_key = clause_type
    else:
        close = difflib.get_close_matches(clause_type, all_keys, n=1, cutoff=0.6)
        if not close:
            return (
                f"No clause type matching '{clause_type}' found.\n"
                f"All available clause types: {sorted(all_keys)}"
            )
        matched_key = close[0]

    matches = [cid for cid, clauses in contract_store.items() if matched_key in clauses]

    note = f" (fuzzy-matched from '{clause_type}')" if matched_key != clause_type else ""
    lines = [f"  {cid}" for cid in matches[:20]]
    suffix = f"\n  ... and {len(matches) - 20} more" if len(matches) > 20 else ""

    return (
        f"Found {len(matches)} contracts with '{matched_key}'{note}.\n"
        f"Pass any of these IDs to search_clauses(contract_id=..., clause_type='{matched_key}'):\n"
        + "\n".join(lines) + suffix
    )

# Smoke test
print(find_contracts_with_clause('limitation_of_liability'))

Found 273 contracts with 'cap_on_liability' (fuzzy-matched from 'limitation_of_liability').
Pass any of these IDs to search_clauses(contract_id=..., clause_type='cap_on_liability'):
  euromediaholdingscorp_20070215_10sb12g_ex_10_b_01
  tomonlineinc_20060501_20_f_ex_4_46_749700_ex_4_46
  gentechholdingsinc_20190808_1_a_ex1a_6_mat_ctrct_1
  theglobecominc_19990503_s_1a_ex_10_20_5416126_ex_1
  liquidmetaltechnologiesinc_20200205_8_k_ex_10_1_11
  playboyenterprisesinc_20090220_10_qa_ex_10_2_40915
  gainscoinc_01_21_2010_ex_10_41_sponsorship_agreeme
  ivillageinc_03_17_1999_ex_10_16_sponsorship_agreem
  nexstarfinanceholdingsinc_03_27_2002_ex_10_26_outs
  mplxlp_06_17_2015_ex_10_1_transportation_services
  senmiaotechnologyltd_02_19_2019_ex_10_5_collaborat
  berkeleylights_inc_06_26_2020_ex_10_12_collaborati
  foundationmedicine_inc_02_02_2015_ex_10_2_collabor
  aurasystemsinc_06_16_2010_ex_10_25_strategic_allia
  entrustinc_07_24_1998_ex_10_5_strategic_alliance_a
  usasyntheticfuelcorp_10_

#### Tool 7: list_clause_types
Returns all clause types that have at least one contract in the store, sorted by
frequency. The agent should call this when it's unsure what clause types are available —
for example, before a task that asks for a clause type that might not exist in the dataset.

In [11]:
def list_clause_types() -> str:
    """List all clause types available in the contract store, sorted by frequency.

    Call this when you're unsure whether a specific clause type exists in the dataset,
    or when a clause type lookup returns no results. Use the returned keys verbatim
    as the clause_type argument in search_clauses() or find_contracts_with_clause().

    Returns a ranked list of clause types with their contract counts.
    """
    counts = Counter(
        clause_type
        for clauses in contract_store.values()
        for clause_type in clauses
    )
    lines = [
        f"  {key} ({count} contracts)"
        for key, count in counts.most_common()
    ]
    return f"Available clause types ({len(counts)} total):\n" + "\n".join(lines)


# Smoke test
print(list_clause_types())

Available clause types (35 total):
  governing_law (425 contracts)
  assignment (371 contracts)
  cap_on_liability (273 contracts)
  license_grant (252 contracts)
  audit_rights (213 contracts)
  post_termination_services (182 contracts)
  termination_for_convenience (180 contracts)
  exclusivity (178 contracts)
  auto_renewal (175 contracts)
  insurance (166 contracts)
  minimum_commitment (163 contracts)
  revenue_sharing (162 contracts)
  non_transferable_license (136 contracts)
  ip_ownership (122 contracts)
  change_of_control (120 contracts)
  non_compete (117 contracts)
  uncapped_liability (111 contracts)
  covenant_not_to_sue (100 contracts)
  rofr_rofo_rofn (85 contracts)
  volume_restriction (81 contracts)
  competitive_restriction_exception (76 contracts)
  warranty (73 contracts)
  irrevocable_perpetual_license (70 contracts)
  liquidated_damages (61 contracts)
  affiliate_license_licensee (59 contracts)
  no_solicit_employees (57 contracts)
  joint_ip_ownership (46 contra

---
## 3. Building the Agent

`dspy.ReAct` implements the **Reason + Act** loop:

```
while not done and iterations < max_iters:
    thought   = lm.reason(goal, history)          # What should I do next?
    action    = lm.pick_tool(thought, tools)       # Which tool? What args?
    observation = tools[action.name](**action.args) # Execute the tool
    history.append((thought, action, observation))
finish = lm.extract_output(goal, history)          # Produce the final structured output
```

The `Signature` defines what the agent is trying to produce. Its docstring is the system prompt
for the whole agent — make it specific about sequencing constraints (e.g., 'always call
list_contracts first') to prevent the agent from hallucinating clause text it hasn't retrieved.

In [12]:
class ContractReviewSignature(dspy.Signature):
    """You are an experienced contract review lawyer conducting a systematic clause review.

    Ground every finding in specific clause text you have retrieved.

    Tool selection guidance:
    - Use list_clause_types() if unsure whether a clause type exists in the dataset.
    - Use find_contracts_with_clause() to get a guaranteed-valid contract ID before
      calling search_clauses() on a clause-first task.
    - Use assess_clause() to evaluate retrieved text against a specific standard.
    - Use web_search() + fetch_url() when you need market data or regulatory guidance.
    """

    task: str = dspy.InputField(
        desc="The review task: what contract(s) to look at, what clause types to review, "
             "what standard to assess against, and what output is expected."
    )
    findings: list[str] = dspy.OutputField(
        desc="Specific findings, each citing the clause text excerpt and the identified issue. "
             "Format: '[Clause type from Contract X]: Issue description — Quote: \"...\"' where 'Contract X' is the contract name."
    )
    sources: list[str] = dspy.OutputField(
        desc="List of sources cited in the findings, including contract names and any urls from web search results that the findings rely on."
    )
    summary: str = dspy.OutputField(
        desc="Executive summary of the review outcome in 2–3 sentences."
    )
    recommendations: list[str] = dspy.OutputField(
        desc="Prioritized action items for counsel, ordered most to least urgent."
    )

TOOLS = [
    list_clause_types,
    list_contracts,
    find_contracts_with_clause,
    search_clauses,
    assess_clause,
    web_search,
    fetch_url,
]

agent = dspy.ReAct(
    ContractReviewSignature,
    tools=TOOLS,
    max_iters=12,  # Enough headroom for the complex tasks
)

print(f'Agent ready — {len(TOOLS)} tools, max 12 iterations')
for t in TOOLS:
    print(f'  {t.__name__}')

Agent ready — 7 tools, max 12 iterations
  list_clause_types
  list_contracts
  find_contracts_with_clause
  search_clauses
  assess_clause
  web_search
  fetch_url


---
## 4. Task 1 — Governing Law Review

**Goal:** Find a software license agreement in the contract store, retrieve its governing law clause,
and assess it against standard US commercial contract conventions.


This is the baseline case — no web research needed. The agent should be able to complete
it in 3–4 iterations using only the contract tools and its own legal reasoning.

In [13]:
TASK_1 = """
Find a software license agreement in the contract database.
Retrieve its governing law clause.
Assess the clause against these standard US commercial contract requirements:
- Must specify a US state (not just 'United States')
- Should specify whether disputes go to state or federal court
- Should include a jury trial waiver for commercial disputes
Produce a finding for each requirement.
"""

with mlflow.start_run(run_name='task1-governing-law-review'):
    result1 = agent(task=TASK_1)
    mlflow.log_metric('findings_count', len(result1.findings))
    mlflow.log_metric('recommendations_count', len(result1.recommendations))

findings_md  = "\n".join(f"{i}. {f}" for i, f in enumerate(result1.findings, 1))
sources_md   = "\n".join(f"- {s}" for s in result1.sources)
recs_md      = "\n".join(f"- {r}" for r in result1.recommendations)

display(Markdown(f"""
## Task 1: Governing Law Review

### Findings
{findings_md}

### Sources
{sources_md}

### Summary
{result1.summary}

### Recommendations
{recs_md}

---
*Trace: MLflow → Traces → `task1-governing-law-review`*
"""))

🏃 View run task1-governing-law-review at: http://localhost:5001/#/experiments/2/runs/7f1b5e7a37bd40359edaba6bfd380a2d
🧪 View experiment at: http://localhost:5001/#/experiments/2



## Task 1: Governing Law Review

### Findings
1. [Governing Law from Summa Four Inc. Software License Agreement]: Compliant - The clause properly specifies a US state by stating the agreement is governed by the laws of the State of New York. — Quote: "This Agreement shall be governed by and interpreted in accordance within the laws of the State of New York without reference to conflicts of laws provisions."
2. [Governing Law from Summa Four Inc. Software License Agreement]: Non-compliant - The clause fails to specify whether disputes must be brought in state or federal court, leaving venue uncertain and potentially allowing forum shopping. — Quote: "This Agreement shall be governed by and interpreted in accordance within the laws of the State of New York without reference to conflicts of laws provisions."
3. [Governing Law from Summa Four Inc. Software License Agreement]: Non-compliant - The clause lacks a jury trial waiver, which is recommended for commercial disputes to avoid unpredictable jury verdicts and ensure specialized commercial resolution. — Quote: "This Agreement shall be governed by and interpreted in accordance within the laws of the State of New York without reference to conflicts of laws provisions."

### Sources
- Summa Four Inc. Software License Agreement (Contract ID: summafourinc_06_19_1998_ex_10_3_software_license_a)

### Summary
The governing law clause in the Summa Four Inc. Software License Agreement complies with the baseline requirement to specify a US state (New York) but falls short of modern commercial contracting standards by omitting venue selection and jury trial waiver provisions. To reduce litigation uncertainty and align with current best practices for enterprise software licensing, the agreement should be supplemented to address these procedural gaps.

### Recommendations
- Amend the governing law provision to add an exclusive venue clause specifying that disputes must be filed in either New York state courts or the federal courts located in New York (Southern or Eastern District), and specify whether both state and federal courts have concurrent jurisdiction or if venue is exclusive to one system.
- Insert a mutual jury trial waiver provision applicable to all commercial disputes arising under the agreement, ensuring both parties waive their constitutional right to a jury trial and consent to bench trial adjudication.
- Consider consolidating all dispute resolution terms (governing law, venue, jury waiver, service of process, and potentially arbitration provisions) into a comprehensive 'Governing Law and Dispute Resolution' article to ensure procedural consistency.

---
*Trace: MLflow → Traces → `task1-governing-law-review`*


Trace(trace_id=tr-9d0a4da950bbb4ff2c99417495fca96e)

---
## 5. Task 2 — Liability Cap vs. Market Standard

**Goal:** Find a limitation of liability clause and compare it to current market practice
in enterprise SaaS agreements — which requires a web search the agent can't avoid.

The agent must research market standards before it can make an assessment —
it can't just apply a hardcoded rule. This forces it to ground its legal opinion in
external sources.

In [14]:
TASK_2 = """
Find a cap on liability clause in any commercial or software contract in the dataset.
Use find_contracts_with_clause('cap_on_liability') to locate a suitable contract first.

Before assessing it, research the current market standard for liability caps in
enterprise SaaS agreements (2024-2025). Specifically look for:
- The most common cap multiple (e.g. 12 months of fees, direct damages only)
- Whether mutual vs. one-sided caps are standard
- Common carve-outs (IP infringement, gross negligence, willful misconduct)

Then assess whether the clause you found aligns with that market standard.
Cite the market research you found to support your assessment.
"""

with mlflow.start_run(run_name='task2-liability-cap-market-standard'):
    result2 = agent(task=TASK_2)
    mlflow.log_metric('findings_count', len(result2.findings))
    mlflow.log_metric('recommendations_count', len(result2.recommendations))

findings_md = "\n".join(f"{i}. {f}" for i, f in enumerate(result2.findings, 1))
sources_md  = "\n".join(f"- {s}" for s in result2.sources)
recs_md     = "\n".join(f"- {r}" for r in result2.recommendations)

display(Markdown(f"""
## Task 2: Liability Cap vs. Market Standard

### Findings
{findings_md}

### Sources
{sources_md}

### Summary
{result2.summary}

### Recommendations
{recs_md}

---
*Trace: MLflow → Traces → `task2-liability-cap-market-standard`*
"""))

🏃 View run task2-liability-cap-market-standard at: http://localhost:5001/#/experiments/2/runs/2ad21fd07ea347a6b33705b17ea477ab
🧪 View experiment at: http://localhost:5001/#/experiments/2



## Task 2: Liability Cap vs. Market Standard

### Findings
1. [Cap on Liability from BERKELEYLIGHTS,INC_06_26_2020-EX-10.12]: The clause provides for mutual limitation of liability (EITHER PARTY... LIABLE TO THE OTHER PARTY), which aligns with the 2024-2025 market standard favoring mutual exclusions over one-sided vendor limitations — Quote: "EXCEPT TO THE EXTENT ARISING (A) FROM A PARTY'S BREACH OF ARTICLE 10 (CONFIDENTIALITY)... IN NO EVENT WILL EITHER PARTY BE LIABLE TO THE OTHER PARTY FOR ANY LOST PROFITS, INDIRECT, SPECIAL, INCIDENTAL, EXEMPLARY OR CONSEQUENTIAL DAMAGES"
2. [Cap on Liability from BERKELEYLIGHTS,INC_06_26_2020-EX-10.12]: The clause properly carves out fraud, gross negligence, and willful misconduct from the liability cap, consistent with market standards requiring unlimited liability for intentional misconduct — Quote: "(E) FROM A PARTY'S FRAUD, GROSS NEGLIGENCE OR WILLFUL MISCONDUCT"
3. [Cap on Liability from BERKELEYLIGHTS,INC_06_26_2020-EX-10.12]: The clause includes indemnification obligations as a carve-out (item F), which partially aligns with standards, though it is unclear if this specifically encompasses IP indemnification — Quote: "(F) IN CONNECTION WITH A PARTY'S INDEMNIFICATION OBLIGATIONS UNDER ARTICLE 12"
4. [Cap on Liability from BERKELEYLIGHTS,INC_06_26_2020-EX-10.12]: The clause lacks an explicit carve-out for IP infringement in the visible text (item B is redacted as [***]), which deviates from the 2024-2025 market standard where IP infringement is a critical carve-out requiring unlimited liability — Quote: "(B) [***]"
5. [Cap on Liability from BERKELEYLIGHTS,INC_06_26_2020-EX-10.12]: The monetary cap amount is redacted/omitted in the retrieved text (<omitted>), preventing verification against the 12-month fees standard — Quote: "ON <omitted> ANY THEORY OF LIABILITY"

### Sources
- BERKELEYLIGHTS,INC_06_26_2020-EX-10.12-COLLABORATI (Contract ID: berkeleylights_inc_06_26_2020_ex_10_12_collaborati)
- https://galkinlaw.com/limitation-of-liability-for-saas/ (Galkin Law, Feb 2024)
- https://www.contractken.com/glossary/limitation-of-liability (ContractKen)
- https://contractnerds.com/negotiating-saas-agreements/ (Contract Nerds)

### Summary
The cap on liability clause in the Berkeley Lights collaboration agreement partially aligns with 2024-2025 enterprise SaaS market standards by providing mutual exclusions of consequential damages and carving out fraud, gross negligence, willful misconduct, confidentiality breaches, and indemnification obligations. However, the clause presents two material concerns: (1) the monetary cap amount is redacted in the retrieved text, preventing verification against the standard 12-month fees benchmark, and (2) the visible text lacks an explicit carve-out for IP infringement—though item (B) is redacted—which is a critical market standard exception where vendors are typically expected to accept unlimited liability.

### Recommendations
- Immediately verify the redacted monetary cap amount to ensure it conforms to the 12-month fees paid standard; if the cap is lower or based on a different metric, negotiate adjustment to meet market standard
- Confirm whether redacted item (B) explicitly covers IP infringement claims; if not, add a specific carve-out for third-party IP infringement claims and associated indemnification to align with 2024-2025 SaaS standards
- Add an explicit carve-out for data security breaches and data loss if not already covered under confidentiality or indemnification provisions, as these are standard unlimited liability exceptions in modern SaaS agreements
- Ensure the indemnification carve-out (Article 12) specifically encompasses IP indemnification obligations, not just general indemnity, to prevent the liability cap from inadvertently limiting IP infringement remedies

---
*Trace: MLflow → Traces → `task2-liability-cap-market-standard`*


Trace(trace_id=tr-5ef99d934d2f88efa90d5de6b6a0ef33)

---
## 6. Task 3 — GDPR Data Deletion Gap Analysis

**Goal:** Review a termination clause against GDPR Article 28(3)(g) — the obligation
to delete or return personal data when a data processing relationship ends.

This is one of the most common gaps in commercial contracts: the termination clause
specifies *when* the agreement ends, but rarely addresses *what happens to personal data
held by the processor* after termination. GDPR Art. 28 requires this to be explicit.

This task is designed to push `max_iters` — the agent needs to synthesize contract text,
regulatory requirements, and practical guidance from multiple sources before it can
make a complete assessment.

In [15]:
TASK_3 = """
Conduct a GDPR data deletion gap analysis on a termination clause from the contract database.
Use find_contracts_with_clause('termination_for_convenience') to locate a suitable contract.

Before assessing the clause, research current GDPR requirements for data deletion
and return when a data processing relationship ends. Look specifically for ICO or
EDPB guidance on Article 28(3)(g) — the processor obligation to:
1. Delete or return all personal data to the controller upon termination
2. Delete existing copies unless EU/member state law requires storage
3. Provide written confirmation of deletion to the controller

Then assess the termination clause against each of the three requirements above.
For each requirement, give a clear verdict (compliant / non_compliant / requires_review)
and, if non-compliant, a specific suggested revision.

The final output should be a gap analysis that a lawyer could hand directly to a client.
"""

with mlflow.start_run(run_name='task3-gdpr-data-deletion-gap-analysis'):
    result3 = agent(task=TASK_3)
    mlflow.log_metric('findings_count', len(result3.findings))
    mlflow.log_metric('recommendations_count', len(result3.recommendations))

findings_md = "\n".join(f"{i}. {f}" for i, f in enumerate(result3.findings, 1))
sources_md  = "\n".join(f"- {s}" for s in result3.sources)
recs_md     = "\n".join(f"- {r}" for r in result3.recommendations)

display(Markdown(f"""
## Task 3: GDPR Data Deletion Gap Analysis

### Findings
{findings_md}

### Sources
{sources_md}

### Summary
{result3.summary}

### Recommendations
{recs_md}

---
*Trace: MLflow → Traces → `task3-gdpr-data-deletion-gap-analysis`*
"""))

🏃 View run task3-gdpr-data-deletion-gap-analysis at: http://localhost:5001/#/experiments/2/runs/c03888f084ca4a74992538798d45b4e9
🧪 View experiment at: http://localhost:5001/#/experiments/2



## Task 3: GDPR Data Deletion Gap Analysis

### Findings
1. [Termination for Convenience from VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT]: VERDICT: NON-COMPLIANT — Requirement 1 (Delete or Return): The clause completely omits any obligation for the processor to delete or return personal data to the controller upon termination. Article 28(3)(g) mandates that the processor must, at the choice of the controller, delete or return all personal data after the end of the provision of services relating to processing. Suggested revision: Add the following termination data handling provision: 'Upon termination or expiry of this Agreement, Supplier shall, at Customer's election, either (i) return all Personal Data to Customer in a structured, commonly used, and machine-readable format within thirty (30) days, or (ii) securely delete all Personal Data from its systems and ensure its Sub-processors do likewise, except where retention is required by applicable Union or Member State law.' — Quote: "Following the Initial Term, Vericel may, without penalty or prejudice to any other rights or remedies Vericel may have, in its sole discretion terminate or reduce the scope of any individual activities contemplated by this Agreement or any Additional Service or with respect to any Product or terminate this Agreement as a whole with or without cause, upon [***] prior written notice of such termination or reduction (which such written notice may be provided during the Initial Term)."
2. [Termination for Convenience from VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT]: VERDICT: NON-COMPLIANT — Requirement 2 (Delete Existing Copies): The clause fails to require deletion of existing copies of personal data from systems, backups, or archives. Article 28(3)(g) explicitly requires that the processor delete existing copies unless Union or Member State law requires storage. Suggested revision: Add: 'Supplier shall delete all existing copies of Personal Data from its active systems, backup systems, and archives within ninety (90) days of termination, except where EU or Member State law requires retention, in which case such Personal Data shall remain subject to the confidentiality obligations and GDPR safeguards of this Agreement until deletion is permissible and shall be deleted promptly once the retention period expires.' — Quote: "Following the Initial Term, Vericel may, without penalty or prejudice to any other rights or remedies Vericel may have, in its sole discretion terminate or reduce the scope of any individual activities contemplated by this Agreement or any Additional Service or with respect to any Product or terminate this Agreement as a whole with or without cause, upon [***] prior written notice of such termination or reduction (which such written notice may be provided during the Initial Term)."
3. [Termination for Convenience from VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT]: VERDICT: NON-COMPLIANT — Requirement 3 (Written Confirmation): The clause lacks any requirement for the processor to provide written confirmation of data deletion to the controller. Per EDPB Guidelines 7/2020, the processor should confirm to the controller when deletion has been completed within an agreed timeframe and manner. Suggested revision: Add: 'Upon completion of the return or deletion of all Personal Data, Supplier shall provide a written certificate signed by an authorized officer within thirty (30) days confirming that (i) all Personal Data has been deleted or returned in accordance with this clause, (ii) no copies remain in Supplier's or its Sub-processors' systems (except as required by law), and (iii) details of any Personal Data retained due to legal requirements, including the applicable legal basis and retention period.' — Quote: "Following the Initial Term, Vericel may, without penalty or prejudice to any other rights or remedies Vericel may have, in its sole discretion terminate or reduce the scope of any individual activities contemplated by this Agreement or any Additional Service or with respect to any Product or terminate this Agreement as a whole with or without cause, upon [***] prior written notice of such termination or reduction (which such written notice may be provided during the Initial Term)."

### Sources
- VERICELCORP_08_06_2019-EX-10.10-SUPPLY AGREEMENT (Contract ID: vericelcorp_08_06_2019_ex_10_10_supply_agreement)
- https://www.bytebacklaw.com/2020/10/analyzing-the-edpbs-guidelines-on-article-28-data-processing-agreements/
- https://www.edpb.europa.eu/system/files/2023-04/si_sa_standard_contract_clauses_en.pdf
- https://www.lexgo.be/en/news-and-articles/7241-new-controller-processor-guidelines-beware-of-impact-on-data-processing-agreements

### Summary
The termination for convenience clause in the Vericel Supply Agreement contains a complete gap regarding GDPR Article 28(3)(g) data deletion obligations. The clause addresses only the mechanics of termination notice but entirely omits any provisions for personal data handling upon termination. All three core requirements—(1) delete/return data at controller's choice, (2) delete existing copies unless legally required, and (3) provide written confirmation—are absent, rendering the contract non-compliant with GDPR processor obligations. Immediate amendment is required to address these material compliance gaps.

### Recommendations
- URGENT: Amend the Supply Agreement immediately to add a comprehensive data deletion clause upon termination that explicitly requires the processor to delete or return all personal data at the controller's election, including a specific timeline (e.g., 30 days) for completion.
- Add specific language requiring deletion of existing copies from all systems (including backups and archives) unless Union or Member State law mandates retention, with explicit safeguards and timelines for any legally retained data.
- Include a written certification requirement mandating the processor provide signed confirmation of deletion completion within 30 days, detailing any exceptions taken due to legal retention requirements and the basis for such retention.
- Conduct a portfolio-wide GDPR Article 28 gap analysis across all processor contracts to identify similar termination clause deficiencies, as this appears to be a systemic drafting omission in legacy agreements.
- Verify whether Vericel currently processes personal data under this active agreement; if so, implement interim data handling protocols and a side-letter amendment pending full contract revision to ensure ongoing compliance.

---
*Trace: MLflow → Traces → `task3-gdpr-data-deletion-gap-analysis`*


Trace(trace_id=tr-a4716917ebee952a55d46f700b7e2c74)